# Build a Transformer from Scratch — the Karpathy way (Part 1: fundamentals)

This is a **step-by-step, from-zero** build of a Transformer language model, following the structure
of Andrej Karpathy's *"Let's build GPT: from scratch, in code, spelled out."*

We start with the simplest possible model and add **one idea at a time**, so that by the end you
understand *why* each part of a Transformer exists. We train a tiny **character-level** model on
text, exactly like the video.

> This is **Part 1: fundamentals**. Once these mechanics click, a follow-up notebook will map them
> onto the **SmolVLA** architecture (`SmolVLA.pdf` in this repo). For now, forget robots — just learn
> the Transformer.

### The plan (each step builds on the last)
1. Get a text dataset and look at it
2. **Tokenize** at the character level (encode/decode)
3. Batching: `block_size`, `batch_size`, and next-token targets
4. **Bigram** baseline — the training/eval/generate skeleton with *no* attention
5. The **mathematical trick** at the heart of self-attention (averaging the past 3 ways)
6. A **single self-attention head**
7. **Multi-head** attention
8. **Feed-forward** network
9. **Blocks**: residual connections + LayerNorm
10. Assemble the full **GPT**, train it, and generate text
11. A short bridge to what comes next (SmolVLA)

Run the cells top to bottom. A GPU (Colab: *Runtime → Change runtime type → GPU*) makes step 10 fast,
but everything runs on CPU too (just slower).


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)                     # same seed Karpathy uses, for reproducibility
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

## 1. Get the data

We use the classic **tiny Shakespeare** corpus (~1 MB of text). We try to download it; if there's no
network, we fall back to a small embedded sample so the notebook still runs.


In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
FALLBACK = (
    "First Citizen:\nBefore we proceed any further, hear me speak.\n\n"
    "All:\nSpeak, speak.\n\n"
    "First Citizen:\nYou are all resolved rather to die than to famish?\n\n"
    "All:\nResolved. resolved.\n\n"
    "First Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\n"
)
try:
    text = urllib.request.urlopen(URL, timeout=15).read().decode("utf-8")
    print("downloaded tiny shakespeare")
except Exception as e:
    print("download failed (", e, ") -> using small embedded fallback")
    text = FALLBACK * 400        # repeat so there's enough to train on

print("length of dataset in characters:", len(text))
print("----- first 250 chars -----")
print(text[:250])

## 2. Tokenize at the character level

The model can't read text — it reads **integers**. The simplest tokenizer maps each unique
*character* to an integer id. We build `stoi` (string→int) and `itos` (int→string) lookup tables,
then `encode`/`decode`.

(Real models use *subword* tokenizers like BPE with ~50k tokens; char-level keeps the vocab tiny and
the idea crystal clear.)


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print("vocabulary:", "".join(chars))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]           # string -> list[int]
decode = lambda l: "".join(itos[i] for i in l)    # list[int] -> string

print(encode("hello"))
print(decode(encode("hello")))

In [ ]:
# Encode the whole dataset into one long tensor, and split 90/10 train/val.
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("train:", train_data.shape, "| val:", val_data.shape)
print("first 40 tokens:", train_data[:40].tolist())

## 3. Batching: context length and next-token targets

A language model is trained to **predict the next token given all previous tokens**. We chop the data
into chunks of length `block_size` (the maximum context). Within a chunk of length `T`, there are `T`
training examples packed together: predict token 1 from token 0, token 2 from tokens 0–1, and so on.

`get_batch` grabs `batch_size` random chunks and returns:
- `x`: the context, shape `(batch_size, block_size)`
- `y`: the same sequence shifted left by one (the targets), same shape


In [ ]:
block_size = 32       # max context length (Karpathy's video uses 8, then 256; 32 is a good middle)
batch_size = 16       # independent sequences processed in parallel

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x shape:", xb.shape, "| y shape:", yb.shape)
print("\nWhat 'predict the next token' means, unpacked for one chunk:")
for t in range(6):
    context = xb[0, : t + 1].tolist()
    target = yb[0, t].item()
    print(f"  context {context}  ->  target {target}")

## 4. Bigram baseline — the whole skeleton, no attention yet

Before any attention, build the simplest model: a **bigram** language model. Each token id directly
looks up a row of logits over the next token via an `nn.Embedding`. It literally only uses the *last*
token to predict the next — no context, no attention. It's a bad model, but it lets us stand up the
**exact training / evaluation / generation loop** we'll reuse for the real Transformer.

Note `generate()` only ever uses the last time step's logits — that's the autoregressive loop.


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token id -> a row of `vocab_size` logits for the next token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)          # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)                          # (B, T, vocab)
            logits = logits[:, -1, :]                      # keep only the last step -> (B, vocab)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, 1)         # sample
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

m = BigramLanguageModel(vocab_size).to(device)
logits, loss = m(xb, yb)
print("logits:", logits.shape, "| initial loss:", loss.item())
print("(a random model should have loss ~= ln(vocab_size) =", round(torch.log(torch.tensor(vocab_size*1.0)).item(), 3), ")")

In [ ]:
# A small reusable helper to estimate loss on train/val without gradient noise.
@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Train the bigram model briefly.
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-2)
for step in range(2000):
    xb, yb = get_batch("train")
    _, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print("bigram loss after training:", estimate_loss(m))

# Generate from the trained bigram model (starting from a single newline token).
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print("----- bigram sample -----")
print(decode(m.generate(start, max_new_tokens=200)[0].tolist()))

The bigram sample is gibberish with the right *character statistics* — it never looks back more than
one token. To do better, a token must be able to **gather information from previous tokens**. That is
exactly what self-attention provides. Next we derive it.


## 5. The mathematical trick at the heart of self-attention

Goal: let each position `t` build a representation from **all positions `≤ t`** (it must not see the
future). Start with the crudest version of "gather from the past": just **average** the previous
tokens. We'll compute this same average three ways — the third way generalizes into attention.


In [ ]:
# Toy tensor: batch=1, time=8, channels=2
torch.manual_seed(1337)
B, T, C = 1, 8, 2
x = torch.randn(B, T, C)

# --- Version 1: explicit loop. xbow[t] = mean of x[0..t]  ('bow' = bag of words) ---
xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, : t + 1].mean(dim=0)

# --- Version 2: the same average as a matrix multiply with a normalized lower-triangular matrix ---
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)      # each row averages the tokens up to and including t
xbow2 = wei @ x                               # (T,T) @ (B,T,C) -> (B,T,C)

print("weight matrix (row t averages cols 0..t):")
print(wei)
print("versions 1 and 2 match:", torch.allclose(xbow, xbow2))

In [ ]:
# --- Version 3: produce those weights with softmax over masked scores ---
# This is the form self-attention uses: start from scores, forbid the future with -inf, softmax.
tril = torch.tril(torch.ones(T, T))
scores = torch.zeros(T, T)                     # here scores are all 0 -> uniform average
scores = scores.masked_fill(tril == 0, float("-inf"))   # block the future
wei3 = F.softmax(scores, dim=-1)
xbow3 = wei3 @ x
print("version 3 matches:", torch.allclose(xbow, xbow3))
print("\nThe key idea: those weights don't have to be uniform.\n"
      "If the scores are *computed from the data*, each token can decide how much to pull\n"
      "from each earlier token. That data-dependent weighting IS self-attention.")

## 6. A single self-attention head

Now make the weights data-dependent. Every token emits:
- a **query** ("what am I looking for?")
- a **key**   ("what do I contain?")
- a **value** ("what will I share if attended to?")

The affinity between position `t` and position `s` is `query_t · key_s`. We mask the future, softmax,
and use the weights to average the **values**. We also scale by `1/sqrt(head_size)` so the softmax
doesn't saturate (Karpathy's "scaled" attention).


In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)        # (B, T, head_size)
q = query(x)      # (B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, T) scaled affinities

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))     # causal mask: no peeking ahead
wei = F.softmax(wei, dim=-1)

v = value(x)      # (B, T, head_size)
out = wei @ v     # (B, T, head_size)

print("output shape:", tuple(out.shape))
print("\nattention weights for one example (row t attends over cols 0..t):")
print(wei[0].round(decimals=2))

Notice the weights are now **different per row and per example** — data-dependent — and strictly
lower-triangular (causal). That single block is the entire innovation. Everything else in a
Transformer is plumbing that makes stacks of these trainable. Let's package it.


In [ ]:
class Head(nn.Module):
    # one self-attention head
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # a non-learned buffer holding the causal mask
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v                                        # (B, T, head_size)

## 7. Multi-head attention

One head captures one kind of relationship. **Multi-head** attention runs several heads in parallel
(each in a smaller subspace), concatenates their outputs, and projects back. This lets the model
attend to different things at once (e.g. one head tracks the previous vowel, another tracks quotes).


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_heads)]
        )
        self.proj = nn.Linear(n_heads * head_size, n_embd)   # mix the concatenated heads
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_heads*head_size)
        return self.dropout(self.proj(out))

## 8. Feed-forward network

Attention lets tokens **communicate**. But after gathering information, each token needs to
**process** it individually. That's the per-position feed-forward MLP: expand → nonlinearity →
contract. It runs on each position independently.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand (the 4x is the Transformer convention)
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),   # project back
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 9. Blocks: residual connections + LayerNorm

A Transformer **block** = multi-head attention (communicate) + feed-forward (compute). Stacking many
raw blocks won't train well, so we add the two tricks that make depth work:

- **Residual connections** (`x = x + sublayer(x)`): give gradients a straight highway to flow back.
- **LayerNorm**: normalize each token's features. We use the modern **pre-norm** placement — normalize
  *before* each sublayer (`x + sublayer(ln(x))`), which is more stable than the original design.


In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_heads, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, n_embd, head_size, block_size, dropout)
        self.ff = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # communicate (with residual)
        x = x + self.ff(self.ln2(x))     # compute     (with residual)
        return x

## 10. Assemble the full GPT — and train it

Now stack it all:
- a **token embedding** table (id → vector) and a **position embedding** table (Karpathy uses a
  learned one rather than sinusoids),
- a stack of `n_layer` Blocks,
- a final LayerNorm and a linear head back to vocab-size logits.

Then reuse the exact same training loop and `generate()` from the bigram model.


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layer, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_heads, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)                       # (B, T, n_embd)
        pos = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                    # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, Cv = logits.shape
            loss = F.cross_entropy(logits.view(B * T, Cv), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]        # crop to the last block_size tokens
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, 1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

In [ ]:
# Hyperparameters — small so it trains quickly. Bump these up on a GPU for better samples.
n_embd, n_heads, n_layer, dropout = 64, 4, 4, 0.1
max_iters, eval_interval, lr = 3000, 500, 3e-4

model = GPTLanguageModel(vocab_size, n_embd, n_heads, n_layer, block_size, dropout).to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e3, "K")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss(model, eval_iters=50)
        print(f"step {it:4d} | train {losses['train']:.3f} | val {losses['val']:.3f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
# Generate! (Small model + tiny training, so expect Shakespeare-flavored gibberish that has learned
# words, spacing, capitalization, and 'Name:' dialogue structure — a huge step up from the bigram.)
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, max_new_tokens=500)[0].tolist()))

## 11. You just built a GPT. What you now understand

- **Tokenization**: text ↔ integers (char-level here; subword/BPE in real models).
- **Self-attention**: data-dependent, causally-masked weighted averaging via Q·K → softmax → ·V.
- **Multi-head**: several attention subspaces in parallel.
- **Blocks**: attention (communicate) + FFN (compute), glued together by **residuals** and
  **LayerNorm** so deep stacks train.
- **The full loop**: embed → blocks → project to logits → cross-entropy → AdamW → autoregressive
  `generate()`.

### Where to go next
- **Scale it up** (Karpathy's finale): `block_size=256, n_embd=384, n_heads=6, n_layer=6`, more iters,
  on a GPU → recognizable Shakespeare.
- **Part 2 (this repo): connect to SmolVLA.** The same attention you just built is the engine of
  SmolVLA's **action expert** (`SmolVLA.pdf`, §3.1). The differences to look for next:
  - it predicts **continuous actions** with a **flow-matching** (regression) objective, not softmax
    over a vocabulary;
  - it adds **cross-attention** so action tokens read **VLM (vision-language) features**, interleaved
    with the **causal self-attention** you built here;
  - inputs are images + a language instruction + a robot **state token**, not characters.

### Exercises
1. Print the attention weights of `Head` for a real batch and read off what a head has learned.
2. Swap the learned position embedding for the sinusoidal one and compare.
3. Turn dropout off — does the val loss get worse (overfitting)?
4. Change the dataset (paste your own text into `FALLBACK`) and retrain.
